# Preprocessing

### Access the dataset from here :

https://drive.google.com/file/d/1XztaPmhMMhBoEp7XuyDGS5Il_kLUlEEl/view?usp=sharing

### Notes:
* This exam consists of a **Regression** problem.  
* The **target** feature is '**cltv**'.
* **Random state** should be taken as **42** wherever applicable.

In [3]:
import pandas as pd

data = pd.read_csv("../OPPE/V1.csv")

data.info()
data.describe()

<class 'pandas.DataFrame'>
RangeIndex: 6257 entries, 0 to 6256
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              6257 non-null   int64  
 1   gender          6257 non-null   str    
 2   area            5873 non-null   str    
 3   qualification   6257 non-null   str    
 4   income          5856 non-null   float64
 5   marital_status  6257 non-null   int64  
 6   vintage         6257 non-null   int64  
 7   claim_amount    5897 non-null   float64
 8   num_policies    6257 non-null   str    
 9   policy          5885 non-null   str    
 10  type_of_policy  6257 non-null   str    
 11  cltv            6257 non-null   int64  
dtypes: float64(2), int64(4), str(6)
memory usage: 586.7 KB


,id,income,marital_status,vintage,claim_amount,cltv
count,6257.000000,5856.000000,6257.000000,6257.000000,5897.000000,6257.000000
mean,44840.267381,13.644030,0.576474,4.611475,4343.455316,97788.084385
std,25677.961183,20.325859,0.494157,2.291584,3323.570342,91457.709902
min,16.000000,0.030000,0.000000,0.000000,0.000000,24876.000000
25%,22477.000000,4.840000,0.000000,3.000000,2359.000000,52356.000000
50%,45028.000000,7.045000,1.000000,5.000000,4079.000000,66288.000000
75%,67353.000000,9.140000,1.000000,6.000000,6093.000000,103380.000000
max,89383.000000,99.840000,1.000000,8.000000,28859.000000,670368.000000


# Metadata

1. **id**-Unique identifier of a customer  
2. **gender**-Gender of the customer   
3. **area**-Area of the customer   
4. **qualification**-Highest Qualification of the customer  
5. **income**-Income earned in a year (in rupees).   
6. **marital_status**- 0:Single, 1: Married
7. **vintage**-No. of years since the first policy date.  
8. **claim_amount**-Total Amount Claimed by the customer (in rupees)
9. **num_policies**-Total no. of policies issued by the customer
10. **policy**-Active policy of the customer
11. **type_of_policy**-Type of active policy
12. **cltv**- Customer life time value. It is the total amount of money a customer is expected to spend with your business, or on your products, during the lifetime of an average business relationship. **[TARGET]**

### Q.2 [Marks: 2] How many total number of features (excluding target variable) are there in the dataset?
Options

A) 1000

B) 11

C) 12

D) 10

**Final Answer:** `11`

**What This Question Is Asking**
This question wants the number of input columns used to predict the target. In machine learning language:
- `cltv` is the output we want to predict
- every other column is an input feature

**Beginner Logic**
Think like this:
1. First find all columns in the dataset.
2. Remove the target column `cltv`.
3. Count what is left.

**Why This Works**
A model takes input features and learns to predict one output. Since `cltv` is the output, it is not counted as an input feature.

**How To Solve In Code**
Use:
- `data.columns` to see all columns
- `data.drop(columns='cltv')` to remove the target
- `.shape[1]` to count the columns

**Exam Memory Trick**
If a question says “excluding target”, always remove the target first before counting.

**What You Should Study**
- What are features and target variables
- `DataFrame.shape`
- `drop()` in pandas
- Basic dataset inspection


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import Lasso, LinearRegression, Ridge, RidgeCV, SGDRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, PolynomialFeatures

categorical_features = ['gender', 'area', 'qualification', 'marital_status', 'num_policies', 'policy', 'type_of_policy']
numeric_features = ['income', 'vintage', 'claim_amount']


def make_base_split(df):
    """Split the data and do the question-specific imputations before Q15."""
    X = df.drop(columns='cltv')
    y = df['cltv']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    X_train = X_train.drop(columns='id').copy()
    X_test = X_test.drop(columns='id').copy()

    income_median = X_train['income'].median()
    policy_mode = X_train['policy'].mode()[0]
    area_mode = X_train['area'].mode()[0]

    X_train['income'] = X_train['income'].fillna(income_median)
    X_test['income'] = X_test['income'].fillna(income_median)

    X_train['policy'] = X_train['policy'].fillna(policy_mode)
    X_test['policy'] = X_test['policy'].fillna(policy_mode)

    X_train['area'] = X_train['area'].fillna(area_mode)
    X_test['area'] = X_test['area'].fillna(area_mode)

    X_train['claim_amount'] = X_train['claim_amount'].fillna(0)
    X_test['claim_amount'] = X_test['claim_amount'].fillna(0)

    return X_train, X_test, y_train, y_test, income_median, policy_mode, area_mode


def make_preprocessed_split(df):
    """Build the post-Q15 design matrix: OHE categories + scaled numeric columns."""
    X_train, X_test, y_train, y_test, *_ = make_base_split(df)

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
            ('num', MinMaxScaler(), numeric_features),
        ]
    )

    X_train_processed = preprocessor.fit_transform(X_train)
    X_test_processed = preprocessor.transform(X_test)
    return X_train_processed, X_test_processed, y_train, y_test, preprocessor

n_features = data.drop(columns='cltv').shape[1]
print(f'Total input features: {n_features}')


### Q.3 [Marks: 2] What are the unique values of feature `Types of Policy` in the dataset?

A) ['Bronze', 'Gold']

B) ['Gold', 'Silver']

C) ['Platinum', 'Gold', 'Silver', 'Bronze]

D) ['Platinum', 'Gold', 'Silver']


**Final Answer:** `['Platinum', 'Gold', 'Silver']`

**What This Question Is Asking**
This question asks for all different category values present in one column: `type_of_policy`.

**Beginner Logic**
Think like this:
1. Select the column.
2. Ignore missing values if any.
3. Find all distinct values.

**Why This Works**
Categorical columns store labels such as `Gold`, `Silver`, etc. To know the possible classes in that column, we use unique values.

**How To Solve In Code**
The most useful commands are:
- `.unique()` gives different values
- `.dropna()` removes missing values first
- `.value_counts()` helps you also see frequency

**Exam Memory Trick**
If the question says “unique values”, immediately think of `.unique()`.

**What You Should Study**
- Difference between categorical and numerical columns
- `.unique()`, `.nunique()`, `.value_counts()`
- Handling missing values in categorical columns


In [ ]:
unique_policies = data['type_of_policy'].dropna().unique().tolist()
print(unique_policies)


### Q.4 [Marks: 3] Which of the following columns have categorical data?[MSQ]

A) income

B) id

C) area

D) claim_amount

E) qualification

**Final Answer:** `area`, `qualification`

**What This Question Is Asking**
You must identify which of the given columns contain category labels instead of actual measurable numeric quantities.

**Beginner Logic**
Ask yourself for each option:
- Is this a number I can do arithmetic on meaningfully?
- Or is it a label/name/category?

`area` is a label like `Urban` or `Rural`.
`qualification` is a label like `Bachelor` or `High School`.
So these are categorical.

**Why This Works**
Categorical data represents groups or names. Numerical data represents amounts, counts, or measurements.

**How To Solve In Code**
You can inspect:
- `data.dtypes`
- `select_dtypes(include='object')`

But always also use common sense, because some columns can be stored as integers but still behave like categories.

**Important Exam Note**
`marital_status` is coded as `0/1`. Even though it looks numeric, conceptually it is categorical because those numbers are labels, not true measured amounts.

**What You Should Study**
- Numerical vs categorical features
- `object`, `int`, `float` in pandas
- Why encoded labels may still be categorical


In [ ]:
option_columns = ['income', 'id', 'area', 'claim_amount', 'qualification']
print(data[option_columns].dtypes)
print('\nCategorical columns from the options:', data[option_columns].select_dtypes(include='object').columns.tolist())


### Q.5 [Marks: 4] Plot the `heatmap` and mark the pair which has the highest positive correlation value. [MCQ]

A) claim_amount & income

B) income & cltv

C) vintage & income.

D) claim_amount & cltv.

**Final Answer:** `claim_amount` and `cltv`

**What This Question Is Asking**
You need to find which pair of numeric variables moves together the most in the positive direction.

**Beginner Logic**
Think like this:
1. Correlation tells how strongly two numeric columns move together.
2. Positive correlation means when one increases, the other tends to increase.
3. We ignore the diagonal values because each column is perfectly correlated with itself.
4. Among the remaining pairs, choose the biggest positive value.

**Why This Works**
A heatmap is just a visual form of the correlation matrix. The strongest positive pair will show the largest positive number off the diagonal.

**How To Solve In Code**
Main ideas:
- `data.corr(numeric_only=True)` creates correlation matrix
- `sns.heatmap(...)` visualizes it
- then inspect the largest non-diagonal correlation

**Exam Memory Trick**
For heatmap questions:
- diagonal is always 1, ignore it
- only compare off-diagonal values

**What You Should Study**
- Pearson correlation
- Positive vs negative correlation
- Heatmaps
- `corr()` in pandas


In [ ]:
numeric_corr = data.corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(numeric_corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

corr_pairs = numeric_corr.where(~np.eye(len(numeric_corr), dtype=bool)).stack().sort_values(ascending=False)
best_pair = corr_pairs.index[0]
best_value = corr_pairs.iloc[0]
print('Highest positive pair:', best_pair)
print('Correlation value:', round(best_value, 4))


### Q.6 [Marks: 2] Which of the following features have `missing` values?[MSQ]

Options:

A) gender

B) area

C) qualification

D) income

E) claim_amount

F) policy

**Final Answer:** `area`, `income`, `claim_amount`, `policy`

**What This Question Is Asking**
You need to identify which columns contain missing data.

**Beginner Logic**
Think like this:
1. Check every column.
2. Count how many missing values are present.
3. Any column with count greater than 0 has missing values.

**Why This Works**
Missing values are usually stored as `NaN`. Pandas can count them column by column.

**How To Solve In Code**
Useful commands:
- `data.isna().sum()`
- keep only columns where result is greater than 0

**Why This Matters In ML**
Most machine learning models cannot directly handle missing values, so missing columns must be treated before model training.

**What You Should Study**
- What `NaN` means
- `isna()` / `isnull()`
- Missing-value counts
- Basic imputation methods


In [ ]:
missing_counts = data.isna().sum()
print(missing_counts[missing_counts > 0])


### Q.7 [Marks: 4] Break the dataset into features(`X`) and label (`y`), where the column `cltv` goes to `y` and the rest of the columns go to `X`. Enter the avg value of `cltv` column? [NAT]


**Final Answer:** `97788.08`

**What This Question Is Asking**
After separating features and label, the question asks for the average value of the target column `cltv`.

**Beginner Logic**
Think like this:
1. `X` contains all input columns.
2. `y` contains only the target column.
3. Then compute the mean of `y`.

**Why This Works**
The average of the target column is simply the arithmetic mean of all its values.

**How To Solve In Code**
- `X = data.drop(columns='cltv')`
- `y = data['cltv']`
- `y.mean()`

**Exam Memory Trick**
When they say label, target, output, dependent variable, they all mean the same thing here: `cltv`.

**What You Should Study**
- Feature matrix `X`
- Target vector `y`
- Mean, median, standard deviation
- Why target is separated from inputs


In [ ]:
X = data.drop(columns='cltv')
y = data['cltv']
print('X shape:', X.shape)
print('y shape:', y.shape)
print('Average cltv:', round(y.mean(), 2))


### Q.8 [Marks : 3] Split the dataset into training and test dataset using `train_test_split` into `70:30` ratio while keeping random_state =42. What is the shape of the training set (X_train) ? [MCQ]


A) (4379, 11)

B) (4392, 13)

C) (4340, 11)

D) (4379, 15)

**Final Answer:** `(4379, 11)`

**What This Question Is Asking**
You need the shape of the training feature matrix after splitting the data into 70% train and 30% test.

**Beginner Logic**
Think like this:
1. Number of rows changes after splitting.
2. Number of columns stays the same because we have not dropped any feature yet.
3. So training rows become 70% of total rows, and columns remain 11.

**Why This Works**
`train_test_split` divides the rows, not the feature definitions.

**How To Solve In Code**
Use:
- `train_test_split(X, y, test_size=0.3, random_state=42)`
- then inspect `X_train.shape`

**Important Concept**
Shape is always `(rows, columns)`.
So `(4379, 11)` means:
- 4379 training examples
- 11 input features

**What You Should Study**
- `train_test_split`
- meaning of `test_size`
- meaning of `random_state`
- interpreting shapes


In [ ]:
X = data.drop(columns='cltv')
y = data['cltv']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print('X_train shape:', X_train.shape)


### Q.9 [Marks: 2] Drop(remove) `id` column from train and test data because it is not useful in model training. Now how many feature columns are remaining in the training dataset? [NAT]

**Final Answer:** `10`

**What This Question Is Asking**
You are asked how many features remain after removing the `id` column.

**Beginner Logic**
Before dropping `id`, there are 11 input features.
After removing one column, 10 remain.

**Why `id` Is Removed**
`id` is just an identifier. It does not describe customer behavior in a meaningful way for prediction.
A model can memorize IDs, which is not useful learning.

**How To Solve In Code**
Use:
- `X_train.drop(columns='id')`
- then check `.shape[1]`

**Exam Memory Trick**
If a column is only an ID, serial number, roll number, or unique key, it is usually removed before training.

**What You Should Study**
- Feature relevance
- Why identifiers are usually bad predictors
- `drop()` and `shape`


In [ ]:
X_train_base, X_test_base, y_train, y_test, *_ = make_base_split(data)
print('Remaining training features:', X_train_base.shape[1])
print(X_train_base.columns.tolist())


### Q.10 [Marks: 2] Compute and write median of the `income` column of X_train while ignoring the missing values. Replace all NaN values in the income column of X_train and X_test by the median  computed from the X_train (upto two decimal). [NAT].

**Final Answer:** `7.04`

**What This Question Is Asking**
You must compute the median of `income` from `X_train` only, then use that same value to fill missing `income` values in both train and test.

**Beginner Logic**
Think carefully here:
1. Split first.
2. Compute median only on training data.
3. Use the training median to fill missing values in both datasets.

**Why Only `X_train`?**
Because if you also use test data while computing the fill value, information from the test set leaks into training. That is called data leakage.

**Why Median And Not Mean?**
Median is safer when data may be skewed or contain outliers.

**How To Solve In Code**
- `income_median = X_train['income'].median()`
- `fillna(income_median)` on both train and test

**Exam Memory Trick**
For preprocessing after split:
- fit/compute on train
- apply to both train and test

**What You Should Study**
- Mean vs median
- Missing-value imputation
- Data leakage
- Why preprocessing is fit on training data only


In [ ]:
X_train_base, X_test_base, y_train, y_test, income_median, _, _ = make_base_split(data)
print('Median income used for imputation:', round(income_median, 2))
print('Missing income in X_train after fill:', X_train_base['income'].isna().sum())
print('Missing income in X_test after fill:', X_test_base['income'].isna().sum())


### Q.11 [Marks: 2] Which is the most frequent value in the `policy` column of X_train? Replace all NaN value in `policy` column of X_train and X_test by most frequent value in X_train [MCQ]

A) 'A'

B) 'B'

C) 'C'

D) None of the above

**Final Answer:** `A`

**What This Question Is Asking**
You must find the most common category in the `policy` column of `X_train` and use it to replace missing values.

**Beginner Logic**
For categorical columns:
- mean is not used
- median is not used
- mode is used

So the question is really asking for the mode of `policy` in training data.

**Why This Works**
`policy` is categorical. The most natural replacement for missing category values is the most frequent category.

**How To Solve In Code**
- `X_train['policy'].mode()[0]`
- fill missing values using that result

**Exam Memory Trick**
Numeric missing values often use mean/median.
Categorical missing values usually use mode.

**What You Should Study**
- Mode in pandas
- Categorical imputation
- Difference between mean/median/mode use cases


In [ ]:
X_train_base, X_test_base, y_train, y_test, _, policy_mode, _ = make_base_split(data)
print('Most frequent policy:', policy_mode)
print(X_train_base['policy'].value_counts())


### Q.12 [Marks: 2] Which is the most frequent value in the `area` column of X_train? Replace all NaN value in `area` column of X_train and X_test by most frequent value from X_train [MCQ]

A) 'Urban'

B) 'Rural'

C) 'Semi-Urban'

D) None of the above

**Final Answer:** `Urban`

**What This Question Is Asking**
Same idea as the previous one, but for the `area` column.
You must find the mode of `area` in training data.

**Beginner Logic**
Since `area` is categorical, we look for the most frequent label.
That label is then used to fill missing values.

**Why This Works**
For categories like `Urban`, `Rural`, etc., the best simple fill strategy is often the most common class.

**How To Solve In Code**
- `X_train['area'].mode()[0]`
- use that value in `fillna()`

**What You Should Study**
- `.mode()`
- categorical preprocessing
- imputation after train-test split


In [ ]:
X_train_base, X_test_base, y_train, y_test, _, _, area_mode = make_base_split(data)
print('Most frequent area:', area_mode)
print(X_train_base['area'].value_counts())


### Q.13 [Marks: 2] Replace all NaN value in claim_amount column of X_train and X_test by 0 (Zero). After Replacing NAN values from claim_amount column what is the standard deviation of claim_amount column in X_train. (correct upto two decimal places) [NAT]

**Final Answer:** `3358.66`

**What This Question Is Asking**
After replacing missing `claim_amount` values with 0, calculate the standard deviation of that column in `X_train`.

**Beginner Logic**
Steps:
1. Replace missing values with zero.
2. Look only at the training column.
3. Compute standard deviation.

**What Standard Deviation Means**
It measures spread.
- small standard deviation: values stay closer to the mean
- large standard deviation: values are more spread out

**Why This Works**
Once missing values are filled, the column becomes fully numeric and standard deviation can be computed directly.

**How To Solve In Code**
- `X_train['claim_amount'] = X_train['claim_amount'].fillna(0)`
- `X_train['claim_amount'].std()`

**What You Should Study**
- Standard deviation concept
- Effect of imputation on data distribution
- Basic descriptive statistics in pandas


In [ ]:
X_train_base, X_test_base, y_train, y_test, *_ = make_base_split(data)
claim_std = X_train_base['claim_amount'].std()
print('Standard deviation of claim_amount:', round(claim_std, 2))


### Q.14 [Marks: 4] Apply `MinMaxScaler` on `income` column of X_train. Compute and write median of `income` column? (correct Upto 2 decimal)[NAT]

**Final Answer:** `0.07`

**What This Question Is Asking**
You must scale the `income` column using MinMaxScaler and then compute the median of the scaled training column.

**Beginner Logic**
MinMaxScaler changes values using this idea:
- minimum becomes 0
- maximum becomes 1
- everything else comes between 0 and 1

After scaling, compute the median again on the transformed values.

**Why This Works**
Scaling changes the numerical range of a feature while preserving its order.
A value that was small before remains relatively small after scaling.

**How To Solve In Code**
- create `MinMaxScaler()`
- `fit_transform()` on training `income`
- `transform()` on test `income`
- then use `.median()` on scaled train column

**Exam Memory Trick**
For scalers:
- fit on train
- transform train and test

**What You Should Study**
- Min-max normalization formula
- `fit`, `transform`, `fit_transform`
- Why scaling helps many ML models


In [ ]:
X_train_base, X_test_base, y_train, y_test, *_ = make_base_split(data)
income_scaler = MinMaxScaler()
X_train_base[['income']] = income_scaler.fit_transform(X_train_base[['income']])
X_test_base[['income']] = income_scaler.transform(X_test_base[['income']])
print('Scaled income median:', round(X_train_base['income'].median(), 2))


## Apply preprocessing on features of X_train and X_test dataset.

### For Categorical Features

* Apply OneHotEncoding from `sklearn` library on all categorical features(object columns). Do Encoding in the order of following list

  `Categorical Features = ['gender', 'area','qualification','marital_status', 'num_policies', 'policy', 'type_of_policy']`

Lets call the transformed caterical feature matrix $X1$

### For Numerical Features

- apply MinMaxScaler and transform the dataset. Do scaling in the order of following list:

  `Numerical Features =  [ 'income', 'vintage', 'claim_amount' ]`


  - Lets call the transformed numerical feature matrix $X2$

### **Concatenate**(One Hot Encoded Features, Scaled Numerical Features)

After combining transformed categorical feature($X_1$) matrix and transformed numerical feature matrix ($X_2$) (side by side in that order), the output will be $X=[X_1 X_2]$

### Hints
* Apply ColumnTransformer to encode categorical columns and scaling on numerical columns with required preprocessor

* Another way is to separately encode all categorical columns and scale numerical columns and do concatenate (`h-stack`) both. keep categorical columns in front of numerical while concatenating.


* The transformed (as desribed by above steps) X_train and X_test, should be considered as X_train and X_test henceforth.


In [ ]:
# Why: from Q15 onward we build the final model matrix exactly as instructed:
# one-hot encode categorical columns, scale all numeric columns, and concatenate them.
# How: ColumnTransformer keeps this reproducible and in the correct column order.
# Study: ColumnTransformer, OneHotEncoder, MinMaxScaler, and end-to-end preprocessing pipelines.

X_train_processed, X_test_processed, y_train, y_test, preprocessor = make_preprocessed_split(data)
print('Processed X_train shape:', X_train_processed.shape)
print('Processed feature names:')
print(preprocessor.get_feature_names_out())


## Q.15 [Marks: 10] How many features you will get after preprocessing? [MCQ]

[Options]

A) 13

B) 20

C) 25

D) 01

**Final Answer:** `20`

**What This Question Is Asking**
After one-hot encoding the categorical columns and scaling the numeric columns, how many columns are in the final model input matrix?

**Beginner Logic**
This is a transformed-feature-count question.
You start with fewer original columns, but encoding creates extra columns.

Here is the idea:
- each category becomes its own binary column in one-hot encoding
- numeric columns remain numeric, just scaled
- then everything is joined side by side

**Why This Works**
Machine learning models usually need numeric input. OneHotEncoder converts categories into machine-readable numeric columns.

**How To Solve In Code**
Use a `ColumnTransformer` with:
- `OneHotEncoder` for categorical columns
- `MinMaxScaler` for numeric columns
Then inspect the shape after transformation.

**What You Should Study**
- One-hot encoding
- Why categorical labels must become numeric
- ColumnTransformer
- Shape after transformation


In [ ]:
print('Number of features after preprocessing:', X_train_processed.shape[1])


# Model Building

### Q.16 [Marks: 5] Apply `SequentialFeatureSelector` transformer with direction= 'forward' with `LinearRegression()` estimator and select 5 features by fitting to the X_train and y_train.

  `Use cv = KFold(n_splits=5,random_state=42,shuffle=True) in SequentialFeatureSelector.`

### Which of the following options represents the correct integer index of the selected features list?


A) [ 6  9 12 13 19]

B) [ 3  6  9 13 19]

C) [ 8  9 12 14 19]

D) [ 1  2  9 13 19]

E) [ 3  7 10 13 19]


**Final Answer:** `[6, 9, 12, 13, 19]`

**What This Question Is Asking**
You must use forward feature selection to choose the best 5 columns from the preprocessed feature matrix.

**Beginner Logic**
Forward selection works like this:
1. Start with no features.
2. Try each remaining feature one by one.
3. Pick the one that improves model performance the most.
4. Repeat until 5 features are selected.

**Why This Works**
Some features help prediction more than others. Feature selection tries to keep useful features and ignore weaker ones.

**Important Detail**
These numbers are not original column names. They are positions (indices) in the transformed feature matrix after encoding and scaling.

**How To Solve In Code**
Use `SequentialFeatureSelector` with:
- `LinearRegression()`
- `direction='forward'`
- `n_features_to_select=5`
- the given `KFold`

**What You Should Study**
- Feature selection methods
- Forward selection idea
- Cross-validation in feature selection
- Difference between original columns and transformed feature indices


In [ ]:
cv = KFold(n_splits=5, random_state=42, shuffle=True)
sfs = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select=5,
    direction='forward',
    cv=cv,
)
sfs.fit(X_train_processed, y_train)
selected_indices = np.where(sfs.get_support())[0]
print('Selected feature indices:', selected_indices)


### Q.17 [Marks: 3] Apply `LinearRegression` on the trainig set(`X_train` and `y_train`). What is the `R2 score` on the test set(`X_test` and `y_test`). ( Upto 4 digits after decimal points) [NAT]


**Final Answer:** `0.1445`

**What This Question Is Asking**
Train a Linear Regression model on the training set and measure how well it predicts the test set using R2 score.

**What R2 Means In Simple Words**
R2 tells how much of the variation in the target is explained by the model.
- closer to 1: better
- around 0: weak model
- below 0: model is worse than a simple baseline

**Beginner Logic**
Steps:
1. Fit the model on training data.
2. Predict on test data.
3. Compare predictions with true `y_test` using R2.

**Why This Works**
We always evaluate on test data to measure how well the model generalizes to unseen examples.

**How To Solve In Code**
- `LinearRegression().fit(X_train_processed, y_train)`
- `.predict(X_test_processed)`
- `r2_score(y_test, predictions)`

**What You Should Study**
- Linear Regression basics
- Train vs test performance
- Meaning of R2 score


In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train_processed, y_train)
y_pred_lr = linear_model.predict(X_test_processed)
print('Test R2 score:', round(r2_score(y_test, y_pred_lr), 4))


### Q.18 [Marks: 6]Using the `LinearRegression` model, compute the `cross-validation scores` for `5 splits` on training data (X_train and y_train) using `cross_val_score`.Enter the maximum value of `𝑅2 score` ( Upto 4 digits after decimal points) obtained.[NAT]

`Use cv = KFold(n_splits=5,random_state=42,shuffle=True) in SequentialFeatureSelector.`

(**Hint**: By default cross_val_score uses LinearRegression's scoring metric, which is  𝑅2 score.)

**Final Answer:** `0.1816`

**What This Question Is Asking**
Instead of one train-test evaluation, this question asks you to evaluate Linear Regression using 5-fold cross-validation on training data.

**Beginner Logic**
Cross-validation means:
1. Split training data into 5 parts.
2. Train on 4 parts and validate on 1 part.
3. Repeat so each part gets used as validation once.
4. Collect all 5 scores.
5. Take the maximum score here because the question asks for the maximum.

**Why This Works**
One single split can be lucky or unlucky. Cross-validation gives a more stable view of model performance.

**How To Solve In Code**
- use `cross_val_score(...)`
- use the same `KFold(n_splits=5, random_state=42, shuffle=True)`
- then compute `max(scores)`

**What You Should Study**
- K-fold cross-validation
- Why CV is more reliable than one split
- Interpreting multiple validation scores


In [ ]:
cv = KFold(n_splits=5, random_state=42, shuffle=True)
cv_scores = cross_val_score(LinearRegression(), X_train_processed, y_train, cv=cv, scoring='r2')
print('Fold R2 scores:', np.round(cv_scores, 4))
print('Maximum R2 score:', round(cv_scores.max(), 4))


### Q.19 [Marks: 5]Apply `Ridge` regression with **random_state=42** with default penalty value on training set(`X_train and y_train`) and calculate the 𝑅2 score on test_set (`X_test and y_test`). What is the correct score ( Upto 4 digits after decimal points)? [NAT]

**Final Answer:** `0.1445`

**What This Question Is Asking**
Train Ridge Regression and measure test R2.

**What Ridge Does**
Ridge is Linear Regression plus L2 regularization.
It adds a penalty that tries to keep coefficients from becoming too large.

**Beginner Logic**
The workflow is the same as Linear Regression:
1. Fit on training data.
2. Predict on test data.
3. Compute R2.

The only difference is the model type.

**Why This Works**
Regularization can reduce overfitting. In this dataset, Ridge gives nearly the same performance as plain Linear Regression.

**How To Solve In Code**
- `Ridge(random_state=42)`
- `.fit(...)`
- `.predict(...)`
- `r2_score(...)`

**What You Should Study**
- Ridge regression
- L2 penalty
- Overfitting vs regularization
- Comparing models using the same metric


In [ ]:
ridge_model = Ridge(random_state=42)
ridge_model.fit(X_train_processed, y_train)
y_pred_ridge = ridge_model.predict(X_test_processed)
print('Ridge test R2 score:', round(r2_score(y_test, y_pred_ridge), 4))


### Q.20: [Marks: 6] Apply `Lasso` regression with **random_state=42** and **regularization rate=0.1** on the training data(`X_train & y_train`). Enter the value of the intercept you got correctly upto 2 digits after decimal points . [NAT]

**Final Answer:** `103168.82`

**What This Question Is Asking**
Fit a Lasso model and report only its intercept.

**What Intercept Means**
In a linear model:
- coefficients multiply the features
- intercept is the starting baseline value

Very roughly, it is the prediction when feature effects are at zero in the transformed space.

**What Lasso Does**
Lasso uses L1 regularization. It can shrink some coefficients strongly and may even make some exactly zero.

**How To Solve In Code**
- create `Lasso(random_state=42, alpha=0.1)`
- fit it
- inspect `.intercept_`

**What You Should Study**
- Linear model equation
- Intercept vs coefficients
- Lasso regression
- L1 regularization and sparsity


In [ ]:
lasso_model = Lasso(random_state=42, alpha=0.1)
lasso_model.fit(X_train_processed, y_train)
print('Lasso intercept:', round(lasso_model.intercept_, 2))


### Q.21 [Marks: 5] Fit SGDRegressor(`random_state=42`) estimator on the training data(`X_train & y_train`) and predict labels for test_data(`X_test`), lets call it as y_test_predict. The parameters are initialized with default values. Calculate and mark the correct mean_absolute_error value between y_test and y_test_predict from the given options. (Correct upto two decimals) [NAT]

**Final Answer:** `52840.15`

**What This Question Is Asking**
Train `SGDRegressor` and compute Mean Absolute Error on the test set.

**What MAE Means**
MAE = average absolute difference between actual value and predicted value.
It tells, on average, how far your prediction is from the true answer.

If MAE is `52840.15`, it means predictions are off by about 52.8k units on average.

**What SGDRegressor Is**
It is a linear model trained using gradient-based optimization instead of the exact closed-form method.

**Beginner Logic**
Steps:
1. Fit SGDRegressor.
2. Predict test values.
3. Compute `mean_absolute_error`.

**Why Scaling Matters Here**
SGD-based models are very sensitive to feature scale. That is why the preprocessing step before this question matters a lot.

**What You Should Study**
- MAE vs MSE vs R2
- SGD idea at a high level
- Why scaling is important for gradient-based methods


In [ ]:
sgd_model = SGDRegressor(random_state=42)
sgd_model.fit(X_train_processed, y_train)
y_pred_sgd = sgd_model.predict(X_test_processed)
print('SGDRegressor MAE:', round(mean_absolute_error(y_test, y_pred_sgd), 2))


### Q.22: [Marks: 6] Using SGDRegressor(random_state=42) as an estimator for exactly 10 iterations. Write the correct R2 score on test data  [NAT] (correct Upto 4 digits)

**Final Answer:** `0.1422`

**What This Question Is Asking**
Train `SGDRegressor` for exactly 10 iterations and compute the test R2 score.

**Beginner Logic**
This is similar to the previous question, but now the model is intentionally stopped after only 10 iterations.
Then we evaluate using R2 instead of MAE.

**Why This Matters**
SGD improves gradually over iterations. If you stop early, the model may not reach its best possible coefficients.

**Expected Idea**
- more iterations usually allow better optimization
- too few iterations may mean incomplete learning

**How To Solve In Code**
Use:
- `SGDRegressor(random_state=42, max_iter=10)`
- fit, predict, then `r2_score`

**What You Should Study**
- Iterations / epochs in optimization
- SGD convergence
- Difference between training longer vs stopping early


In [ ]:
sgd_model_10 = SGDRegressor(random_state=42, max_iter=10)
sgd_model_10.fit(X_train_processed, y_train)
y_pred_sgd_10 = sgd_model_10.predict(X_test_processed)
print('SGDRegressor (10 iterations) R2:', round(r2_score(y_test, y_pred_sgd_10), 4))


# (Common Instructions for Question 23 and 24)

### Create a pipeline Using PolynomialFeatures as transformer and Lasso as estimator. Use GridSearchCV with this created pipeline and following hyperparameter values on training data(X_train, y_train) to fit the model .
```
1. Keep polynomial degree as : [1, 2]
2. alpha value to be taken as : np.logspace(-3, 0, num=5)
3. scoring : neg_mean_absolute_error .
```
(**Note**: Kindly ignore the warning.)

In [ ]:
# Why: GridSearchCV checks every (degree, alpha) combination using the requested scoring rule.
# How: we wrap PolynomialFeatures and Lasso inside a Pipeline so the search applies both steps cleanly.
# Study: model selection, hyperparameter tuning, pipelines, and custom scoring metrics.

pipeline = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('lasso', Lasso(random_state=42)),
])

param_grid = {
    'poly__degree': [1, 2],
    'lasso__alpha': np.logspace(-3, 0, num=5),
}

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=5,
    n_jobs=-1,
)

grid_search.fit(X_train_processed, y_train)
print('Best parameters:', grid_search.best_params_)
print('Best CV score:', grid_search.best_score_)


### Q.23 [Marks: 6] Mark the best `alpha` value you got using above instructions.[MCQ]

A) 0.001

B) 0.00562341

C) 0.03162278

D) 0.17782794

E) 1.00



**Final Answer:** `1.0`

**What This Question Is Asking**
After trying multiple `alpha` values using GridSearchCV, which `alpha` gave the best score?

**What `alpha` Means**
`alpha` controls regularization strength in Lasso.
- small alpha: weaker penalty
- large alpha: stronger penalty

**Beginner Logic**
Grid search tries every combination in the parameter grid.
Then it compares all combinations using the chosen scoring metric.
The best one is stored in `best_params_`.

**Why This Works**
Instead of guessing hyperparameters manually, grid search systematically checks them.

**How To Solve In Code**
- build `Pipeline([PolynomialFeatures, Lasso])`
- use `GridSearchCV`
- read `grid_search.best_params_['lasso__alpha']`

**What You Should Study**
- Hyperparameters vs learned parameters
- GridSearchCV
- Lasso regularization strength
- Reading `best_params_`


In [ ]:
print('Best alpha:', grid_search.best_params_['lasso__alpha'])


### Q.24 [Marks: 6] Enter the best polynomial degree value you got using above instructions.[NAT]




**Final Answer:** `1`

**What This Question Is Asking**
Among polynomial degree values `[1, 2]`, which one gave the best cross-validated MAE in the grid search?

**What Degree Means**
- degree 1: no extra polynomial interactions, basically linear features
- degree 2: adds squared terms and interactions

**Beginner Logic**
More complex features do not always help. Degree 2 can overfit or simply fail to improve enough.
In this case, degree 1 performs best.

**Why This Works**
Grid search is comparing model performance, not model complexity. The simpler option wins if it performs better.

**How To Solve In Code**
Read:
- `grid_search.best_params_['poly__degree']`

**What You Should Study**
- PolynomialFeatures
- Interaction terms
- Why more complexity is not always better
- Bias-variance tradeoff


In [ ]:
print('Best polynomial degree:', grid_search.best_params_['poly__degree'])


# (Common Instructions for Question 25 and 26)
### To Reduce number of dimensions of training data with PCA. Fit the PCA model using following parameter values on training data.
```
n_components=5
svd_solver='full'
whiten=True
random_state=42
```

In [ ]:
# Why: PCA compresses the 20-feature matrix into 5 orthogonal directions that retain most of the variance.
# How: fit PCA on the processed training matrix, then transform both train and test with the same fitted PCA object.
# Study: principal components, explained variance, whitening, and dimensionality reduction.

pca = PCA(n_components=5, svd_solver='full', whiten=True, random_state=42)
X_train_pca = pca.fit_transform(X_train_processed)
X_test_pca = pca.transform(X_test_processed)
print('PCA train shape:', X_train_pca.shape)
print('Explained variance ratios:', np.round(pca.explained_variance_ratio_, 4))


### Q.25 [Marks: 5] What is the sum of `explained_variance_ratio_` ? [NAT]

**Final Answer:** `0.6592`

**What This Question Is Asking**
After reducing the dataset to 5 principal components using PCA, how much total variance is preserved?

**What PCA Does**
PCA creates new columns called principal components.
These are combinations of the old columns and are chosen to keep as much information as possible.

**What Explained Variance Ratio Means**
It tells how much information each principal component keeps.
When we sum the first 5 ratios, we get total retained variance.

**Beginner Logic**
1. Fit PCA with 5 components.
2. Look at `explained_variance_ratio_`.
3. Add those values.

**Why This Works**
The sum tells how much of the original dataset variation is still present after compression.

**What You Should Study**
- PCA intuition
- Principal components
- Explained variance ratio
- Dimensionality reduction


In [ ]:
print('Sum of explained_variance_ratio_:', round(pca.explained_variance_ratio_.sum(), 4))


### Q.26 [Marks: 6] Use PCA transformed training data from earlier question and y_train to fit the `RidgeCV` estimator model having `alpha value as [0.001,0.01,0.1,1]`. Calculate the R2 score you got from the model for transformed test data(PCA transformed X_test). [NAT] (upto 4 decimal)

**Final Answer:** `0.1299`

**What This Question Is Asking**
Use the PCA-transformed training data to train `RidgeCV`, then evaluate R2 on the PCA-transformed test data.

**Beginner Logic**
This is a 2-step pipeline idea:
1. First compress the features with PCA.
2. Then train a regression model on those compressed features.
3. Use `RidgeCV` to choose the best alpha from the given list.
4. Predict on transformed test data and compute R2.

**Why This Works**
PCA reduces dimensionality, and RidgeCV handles regularized regression while automatically choosing the best alpha.

**Important Exam Idea**
Whenever PCA is fit on training data:
- use `fit_transform()` on train
- use `transform()` on test
Never fit PCA separately on test data.

**What You Should Study**
- PCA workflow
- RidgeCV
- Automatic alpha selection
- End-to-end preprocessing + modeling sequence


In [ ]:
ridge_cv_model = RidgeCV(alphas=[0.001, 0.01, 0.1, 1])
ridge_cv_model.fit(X_train_pca, y_train)
y_pred_ridge_cv = ridge_cv_model.predict(X_test_pca)
print('Chosen alpha:', ridge_cv_model.alpha_)
print('RidgeCV on PCA features R2:', round(r2_score(y_test, y_pred_ridge_cv), 4))
